In [5]:
# Neural Conjecture Proposer (Enhanced Version)
# An advanced tool to generate mathematical conjectures with a pro-level, mathematically themed UI/UX
# Designed for use in Google Colab with Gradio

import os
import time
import re
import json
import random
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime
from tqdm.notebook import tqdm
import pandas as pd
import gradio as gr
import torch
import argparse

# Install required packages if missing
try:
    from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
except ImportError:
    print("Installing required packages...")
    import subprocess
    subprocess.check_call(["pip", "install", "transformers", "torch", "gradio", "matplotlib", "pandas"])
    from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM

# Model options
DEFAULT_MODEL = "distilgpt2"
ALTERNATIVE_MODELS = ["EleutherAI/gpt-neo-125M", "bigscience/bloom-560m"]

# Prompt templates (unchanged)
PROMPT_TEMPLATES = {
    "few-shot": """Here are some known mathematical identities:

1. Sum of natural numbers:
   ∑_{i=1}^n i = n(n+1)/2

2. Sum of squares:
   ∑_{i=1}^n i^2 = n(n+1)(2n+1)/6

3. Sum of cubes:
   ∑_{i=1}^n i^3 = [n(n+1)/2]^2

4. Fibonacci recurrence:
   F(n) = F(n-1) + F(n-2)

5. Catalan numbers:
   C(n) = (1/(n+1)) * (2n choose n)

Now, based on the patterns and structures above, propose a new mathematical conjecture that seems reasonable and follows similar algebraic forms. Your output should be in LaTeX format and include a one-line explanation or insight.

Conjecture:""",

    "symbolic-pattern": """You are an advanced AI trained on mathematical theory. You will now propose a novel formula or recurrence based on observed symbolic structures.

Known patterns:

- Linear recurrences:
  F(n) = aF(n-1) + bF(n-2)

- Polynomial summation:
  ∑_{k=1}^n k^m = P(n)

- Binomial relationships:
  (a + b)^n = ∑_{k=0}^n C(n,k) * a^{n-k} * b^k

Task:
1. Combine any two or more ideas above (e.g., recurrence + binomial coefficient).
2. Propose a symbolic conjecture that fits the patterns.
3. Output in LaTeX syntax.
4. Provide a one-line explanation or insight.

New Conjecture:""",

    "sequence": """Given the following number sequences and closed-form formulas:

A. 1, 3, 6, 10, 15 → ∑_{i=1}^n i = n(n+1)/2
B. 1, 4, 10, 20, 35 → ∑_{i=1}^n i(i+1)/2
C. 1, 5, 14, 30, 55 → ?

Task:
- Identify the likely rule or generating function for sequence C.
- Propose a symbolic conjecture using summation or polynomial form.
- Use LaTeX notation.

Conjecture for C:""",

    "reverse-engineering": """You are a theorem-generating AI. Here is a set of formulas from mathematical literature:

1. ∑_{i=1}^n i^2 = n(n+1)(2n+1)/6
2. ∑_{i=1}^n i^3 = [n(n+1)/2]^2
3. Let's define T(n) as the nth triangular number: T(n) = n(n+1)/2

Use these to propose a **new conjecture** that may connect triangular numbers to cube numbers or higher powers.

Output a new symbolic identity (conjecture) and describe in one line how it's structurally inspired.

Conjecture:""",

    "creative": """You are a creative mathematical AI. Inspired by physics and combinatorics, invent a new identity that blends:

- A recurrence or summation
- A physical constant (like π or e)
- A combinatorial term (like C(n,k) or factorials)

Be creative but syntactically correct. Output in LaTeX format and give a one-line explanation of its structure.

Conjecture:"""
}

# Educational math concepts (unchanged)
MATH_CONCEPTS = {
    "Summation": """The summation operator (∑) represents adding up a sequence of numbers.

**Example**: ∑_{i=1}^5 i = 1 + 2 + 3 + 4 + 5 = 15

**Applications**: Used in calculus, statistics, and discrete mathematics to represent series and sequences.

**Visual Representation**: Triangular numbers can be visualized as dots arranged in triangles, demonstrating summation principles.""",

    "Recurrence Relations": """A recurrence relation defines each term of a sequence in terms of preceding terms.

**Example**: The Fibonacci sequence F(n) = F(n-1) + F(n-2), with F(0) = 0, F(1) = 1

**Applications**: Used in algorithm analysis, population modeling, and solving difference equations.

**Visual Representation**: Recursive tree structures demonstrate how recurrence relations expand.""",

    "Binomial Coefficients": """The binomial coefficient (n choose k) represents the number of ways to choose k items from n items.

**Example**: (5 choose 2) = 10, meaning there are 10 ways to select 2 items from a set of 5

**Applications**: Probability theory, combinatorics, and statistical analysis.

**Visual Representation**: Pascal's Triangle, where each number is the sum of the two numbers above it.""",

    "Polynomial Identities": """Polynomial identities are equations that relate polynomial expressions that are equal for all values of their variables.

**Example**: (a + b)² = a² + 2ab + b²

**Applications**: Algebraic simplification, calculus integration, and number theory.

**Visual Representation**: Geometric interpretations like area models for quadratic expansions.""",

    "Number Sequences": """Number sequences are ordered lists of numbers following a pattern or formula.

**Examples**:
- Arithmetic sequence: 2, 4, 6, 8, ... (add 2)
- Geometric sequence: 2, 4, 8, 16, ... (multiply by 2)

**Applications**: Found in growth patterns, financial models, and mathematical induction.

**Visual Representation**: Graphs showing the growth behavior of different sequence types."""
}

# Advanced CSS with Mathematical Theme
custom_css = """
body {
    background-color: #f0f2f5;
    font-family: 'Times New Roman', serif;
    color: #2c3e50;
    background-image: url('https://www.transparenttextures.com/patterns/graphy-dark.png');
    background-attachment: fixed;
    background-size: cover;
}

.container {
    max-width: 1300px;
    margin: 0 auto;
    padding: 20px;
}

.hero {
    background: linear-gradient(120deg, #2c3e50, #34495e, #2c3e50);
    padding: 80px 20px;
    text-align: center;
    position: relative;
    overflow: hidden;
    border-bottom: 6px solid #ecf0f1;
    box-shadow: 0 4px 12px rgba(0,0,0,0.2);
    animation: heroGradient 15s ease infinite;
}

@keyframes heroGradient {
    0% { background-position: 0% 50%; }
    50% { background-position: 100% 50%; }
    100% { background-position: 0% 50%; }
}

.hero h1 {
    font-size: 52px;
    color: #ecf0f1;
    margin-bottom: 15px;
    text-shadow: 3px 3px 6px rgba(0,0,0,0.5);
    animation: fadeIn 2s ease-in;
}

.hero p {
    font-size: 26px;
    color: #bdc3c7;
    text-shadow: 1px 1px 3px rgba(0,0,0,0.3);
}

.math-decoration {
    position: absolute;
    font-size: 36px;
    color: #ecf0f1;
    opacity: 0.15;
    animation: floatMath 8s ease-in-out infinite;
}

@keyframes floatMath {
    0% { transform: translateY(0) rotate(0deg); }
    50% { transform: translateY(-30px) rotate(5deg); }
    100% { transform: translateY(0) rotate(0deg); }
}

.tab-content {
    background-color: #fff;
    padding: 40px;
    border-radius: 12px;
    box-shadow: 0 6px 18px rgba(0,0,0,0.1);
    margin: 25px 0;
    border: 1px solid #ecf0f1;
}

.gr-button {
    background-color: #2c3e50;
    color: #ecf0f1;
    border: none;
    padding: 14px 28px;
    border-radius: 10px;
    transition: all 0.4s ease;
    font-family: 'Times New Roman', serif;
    font-size: 16px;
    position: relative;
    overflow: hidden;
}

.gr-button:hover {
    background-color: #34495e;
    transform: translateY(-2px);
    box-shadow: 0 4px 12px rgba(0,0,0,0.2);
}

.gr-button::after {
    content: '➔';
    position: absolute;
    right: 15px;
    opacity: 0;
    transition: all 0.3s ease;
}

.gr-button:hover::after {
    opacity: 1;
    right: 10px;
}

.conjecture-card {
    background-color: #f9fbfc;
    padding: 25px;
    border-radius: 10px;
    box-shadow: 0 4px 10px rgba(0,0,0,0.05);
    margin-bottom: 20px;
    transition: all 0.3s ease;
    border-left: 5px solid #3498db;
}

.conjecture-card:hover {
    transform: translateY(-5px);
    box-shadow: 0 8px 16px rgba(0,0,0,0.1);
}

.formula {
    background-color: #ecf0f1;
    padding: 20px;
    border-radius: 8px;
    margin: 15px 0;
    overflow-x: auto;
    font-family: 'Times New Roman', serif;
    font-size: 18px;
    border: 1px dashed #bdc3c7;
}

.error {
    color: #e74c3c;
    padding: 15px;
    background-color: #f9ebeb;
    border-radius: 6px;
    border: 1px solid #e74c3c;
}

.input-group {
    background-color: #fdfdfd;
    padding: 20px;
    border-radius: 10px;
    margin-bottom: 20px;
    box-shadow: inset 0 2px 4px rgba(0,0,0,0.05);
}

.footer {
    text-align: center;
    padding: 30px;
    color: #7f8c8d;
    font-size: 16px;
    border-top: 2px solid #ecf0f1;
    background: linear-gradient(to top, #f0f2f5, #fff);
}

.custom-slider {
    --slider-color: #2c3e50;
    --track-color: #bdc3c7;
}

.gr-slider input[type="range"] {
    -webkit-appearance: none;
    appearance: none;
    background: var(--track-color);
    height: 8px;
    border-radius: 4px;
}

.gr-slider input[type="range"]::-webkit-slider-thumb {
    -webkit-appearance: none;
    appearance: none;
    width: 20px;
    height: 20px;
    background: var(--slider-color);
    border-radius: 50%;
    cursor: pointer;
    transition: all 0.3s ease;
}

.gr-slider input[type="range"]::-webkit-slider-thumb:hover {
    transform: scale(1.2);
}

.analytics-dashboard {
    display: flex;
    flex-wrap: wrap;
    gap: 20px;
    justify-content: space-between;
}

.analytics-card {
    flex: 1 1 300px;
    background-color: #fff;
    padding: 20px;
    border-radius: 10px;
    box-shadow: 0 4px 10px rgba(0,0,0,0.05);
    border-left: 5px solid #2ecc71;
}
"""

# ConjectureGenerator class (unchanged core functionality)
class ConjectureGenerator:
    def __init__(self, model_name=DEFAULT_MODEL):
        self.model_name = model_name
        self.output_dir = Path("conjecture_outputs")
        self.output_dir.mkdir(exist_ok=True)
        self.log_file = self.output_dir / "conjectures_log.json"
        if not self.log_file.exists():
            with open(self.log_file, 'w') as f:
                json.dump([], f)
        self.stats_file = self.output_dir / "generation_stats.json"
        if not self.stats_file.exists():
            with open(self.stats_file, 'w') as f:
                json.dump({
                    "total_generated": 0,
                    "generation_history": [],
                    "model_usage": {},
                    "prompt_usage": {}
                }, f)
        print(f"Loading model: {model_name}")
        try:
            self.tokenizer = AutoTokenizer.from_pretrained(model_name)
            self.model = AutoModelForCausalLM.from_pretrained(model_name)
            self.generator = pipeline('text-generation', model=self.model, tokenizer=self.tokenizer)
            print(f"Model {model_name} loaded successfully!")
        except Exception as e:
            print(f"Error loading model: {e}")
            print("Falling back to distilgpt2...")
            self.model_name = "distilgpt2"
            self.tokenizer = AutoTokenizer.from_pretrained("distilgpt2")
            self.model = AutoModelForCausalLM.from_pretrained("distilgpt2")
            self.generator = pipeline('text-generation', model=self.model, tokenizer=self.tokenizer)

    def generate_conjecture(self, prompt_type="few-shot", custom_prompt=None, temperature=0.9, creativity=0.8):
        start_time = time.time()
        if custom_prompt:
            prompt = custom_prompt
        else:
            if prompt_type not in PROMPT_TEMPLATES:
                prompt_type = "few-shot"
            prompt = PROMPT_TEMPLATES[prompt_type]
        try:
            max_new_tokens = int(100 + (creativity * 100))
            temp = temperature * (0.7 + (creativity * 0.6))
            top_p = 0.92 + (creativity * 0.08)
            gen_params = {
                "max_new_tokens": max_new_tokens,
                "temperature": temp,
                "top_p": top_p,
                "do_sample": True,
                "no_repeat_ngram_size": 2
            }
            output = self.generator(prompt, **gen_params)
            generated_text = output[0]['generated_text']
            conjecture = generated_text[len(prompt):].strip() if generated_text.startswith(prompt) else generated_text.strip()
            clean_conjecture = self._clean_output(conjecture)
            validation = self.validate_conjecture(clean_conjecture)
            if validation["overall_quality"] == "Poor":
                clean_conjecture = self.post_process_conjecture(clean_conjecture)
                validation = self.validate_conjecture(clean_conjecture)
            latex_formula = self._extract_latex(clean_conjecture)
            generation_time = time.time() - start_time
            conjecture_data = self._save_conjecture(
                prompt_type, prompt, clean_conjecture, latex_formula, validation, generation_time,
                {"temperature": temp, "top_p": top_p, "max_tokens": max_new_tokens}
            )
            return conjecture_data
        except Exception as e:
            return {"error": str(e), "status": "failed"}

    def _clean_output(self, text):
        lines = text.split('\n')
        clean_lines = []
        empty_line_count = 0
        for line in lines:
            if not line.strip():
                empty_line_count += 1
                if empty_line_count >= 2:
                    break
            else:
                empty_line_count = 0
            clean_lines.append(line)
            if len(clean_lines) >= 12:
                break
        clean_text = '\n'.join(clean_lines).strip()
        open_delimiters = clean_text.count('\\[') + clean_text.count('$$') + clean_text.count('$')
        close_delimiters = clean_text.count('\\]') + clean_text.count('$$') + clean_text.count('$')
        if open_delimiters > close_delimiters:
            last_complete = clean_text.rfind('\\]')
            if last_complete == -1:
                last_complete = clean_text.rfind('$$')
            if last_complete == -1:
                return clean_text
            return clean_text[:last_complete+2]
        return clean_text

    def _save_conjecture(self, prompt_type, prompt, conjecture, latex_formula, validation, generation_time, params):
        timestamp = int(time.time())
        with open(self.log_file, 'r') as f:
            conjectures = json.load(f)
        conjecture_data = {
            "id": len(conjectures) + 1,
            "timestamp": timestamp,
            "date": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "prompt_type": prompt_type,
            "prompt": prompt,
            "full_response": conjecture,
            "latex_formula": latex_formula,
            "model": self.model_name,
            "validation": validation,
            "generation_time": generation_time,
            "parameters": params
        }
        conjectures.append(conjecture_data)
        with open(self.log_file, 'w') as f:
            json.dump(conjectures, f, indent=2)
        conjecture_file = self.output_dir / f"conjecture_{timestamp}.json"
        with open(conjecture_file, "w") as f:
            json.dump(conjecture_data, f, indent=2)
        return conjecture_data

    def _extract_latex(self, text):
        patterns = [r'\\[.*?\\]', r'\$\$(.*?)\$\$', r'\$(.*?)\$']
        for pattern in patterns:
            matches = re.findall(pattern, text, re.DOTALL)
            if matches:
                return matches[0] if pattern != r'\\[.*?\\]' else matches[0][2:-2]
        lines = text.strip().split('\n')
        for line in enumerate(lines):
            if any(symbol in line for symbol in ["\\sum", "\\prod", "\\int", "="]):
                return line.strip()
        return None

    def batch_generate(self, num_conjectures=5, prompt_types=None, temperature=0.9, creativity=0.8):
        if prompt_types is None:
            prompt_types = list(PROMPT_TEMPLATES.keys())
        results = []
        for _ in tqdm(range(num_conjectures)):
            prompt_type = random.choice(prompt_types)
            result = self.generate_conjecture(prompt_type, temperature=temperature, creativity=creativity)
            if result and "error" not in result:
                results.append(result)
        return results

    def analyze_conjectures(self):
        try:
            with open(self.log_file, 'r') as f:
                conjectures = json.load(f)
            if not conjectures:
                return {"status": "No conjectures found"}
            total_count = len(conjectures)
            prompt_counts = {pt: 0 for pt in PROMPT_TEMPLATES}
            for c in conjectures:
                prompt_type = c.get("prompt_type", "unknown")
                prompt_counts[prompt_type] = prompt_counts.get(prompt_type, 0) + 1
            model_counts = {}
            for c in conjectures:
                model = c.get("model", "unknown")
                model_counts[model] = model_counts.get(model, 0) + 1
            quality_counts = {"Good": 0, "Poor": 0}
            for c in conjectures:
                quality = c.get("validation", {}).get("overall_quality", "unknown")
                if quality in quality_counts:
                    quality_counts[quality] += 1
            recent = sorted(conjectures, key=lambda x: x.get("timestamp", 0), reverse=True)[:5]
            generation_times = [c.get("generation_time", 0) for c in conjectures if "generation_time" in c]
            avg_generation_time = sum(generation_times) / len(generation_times) if generation_times else 0
            return {
                "status": "success",
                "total_count": total_count,
                "prompt_counts": prompt_counts,
                "model_counts": model_counts,
                "quality_counts": quality_counts,
                "recent_conjectures": recent,
                "avg_generation_time": avg_generation_time
            }
        except Exception as e:
            return {"status": "error", "message": str(e)}

    def validate_conjecture(self, conjecture_text):
        math_symbols = ["=", "\\sum", "\\prod", "\\int", "\\frac", "\\cdot", "\\pi", "\\in", "\\subset"]
        has_math_symbols = any(symbol in conjecture_text for symbol in math_symbols)
        symbol_count = sum(conjecture_text.count(symbol) for symbol in math_symbols)
        has_latex = ("\\(" in conjecture_text or "\\[" in conjecture_text or
                     "$" in conjecture_text or "\\begin" in conjecture_text)
        text_length = len(conjecture_text)
        has_good_length = text_length > 20
        has_explanation = any(word in conjecture_text.lower() for word in
                              ["explain", "because", "represents", "where", "note that", "observe"])
        complexity_score = min(100, (
            (20 if has_math_symbols else 0) +
            (20 if has_latex else 0) +
            min(30, symbol_count * 5) +
            min(20, text_length // 10) +
            (10 if has_explanation else 0)
        ))
        overall_quality = "Good" if (has_math_symbols and has_good_length and complexity_score > 40) else "Poor"
        return {
            "has_math_symbols": has_math_symbols,
            "has_latex": has_latex,
            "has_good_length": has_good_length,
            "has_explanation": has_explanation,
            "complexity_score": complexity_score,
            "overall_quality": overall_quality
        }

    def post_process_conjecture(self, conjecture):
        latex_fixes = [
            (r'\\sum([^_])', r'\\sum_{\1}'),
            (r'(?<!\$)\$(?!\$)', r'$$'),
            (r'\\frac([^{])', r'\\frac{\1}'),
            (r'([0-9])([a-zA-Z])', r'\1 \2'),
        ]
        processed = conjecture
        for pattern, replacement in latex_fixes:
            processed = re.sub(pattern, replacement, processed)
        if '=' in processed and not (re.search(r'\\[', processed) or '$' in processed):
            formula_part = re.search(r'([^.]*=[^.]*)', processed)
            if formula_part:
                replacement = f"$${formula_part.group(1)}$$"
                processed = processed.replace(formula_part.group(1), replacement)
        if not any(word in processed.lower() for word in ["where", "note", "observe", "represents", "explain"]):
            processed += "\n\nNote: This conjecture explores relationships in mathematical patterns."
        return processed

    def get_visualization_data(self):
        try:
            with open(self.stats_file, 'r') as f:
                stats = json.load(f)
            with open(self.log_file, 'r') as f:
                conjectures = json.load(f)
            date_counts = {}
            for entry in stats.get("generation_history", []):
                date = entry.get("date", "unknown")
                date_counts[date] = date_counts.get(date, 0) + 1
            model_usage = stats.get("model_usage", {})
            prompt_usage = stats.get("prompt_usage", {})
            quality_counts = {"Good": 0, "Poor": 0}
            for c in conjectures:
                quality = c.get("validation", {}).get("overall_quality", "unknown")
                if quality in quality_counts:
                    quality_counts[quality] += 1
            complexity_scores = [c.get("validation", {}).get("complexity_score", 0) for c in conjectures]
            generation_times = [{"index": i+1, "time": c["generation_time"]}
                               for i, c in enumerate(conjectures) if "generation_time" in c]
            return {
                "date_counts": date_counts,
                "model_usage": model_usage,
                "prompt_usage": prompt_usage,
                "quality_counts": quality_counts,
                "complexity_scores": complexity_scores,
                "generation_times": generation_times,
                "total_generated": stats.get("total_generated", 0)
            }
        except Exception as e:
            return {"status": "error", "message": str(e)}

    def get_recent_conjectures(self, limit=5):
        try:
            with open(self.log_file, 'r') as f:
                conjectures = json.load(f)
            sorted_conjectures = sorted(conjectures, key=lambda x: x.get("timestamp", 0), reverse=True)
            return sorted_conjectures[:limit]
        except Exception as e:
            return []

    def create_visualizations(self):
        data = self.get_visualization_data()
        fig, axs = plt.subplots(2, 2, figsize=(16, 12))
        fig.suptitle('Neural Conjecture Proposer Analytics', fontsize=20, fontfamily='Times New Roman')

        # Prompt Distribution
        prompt_labels = list(data["prompt_usage"].keys())
        prompt_values = list(data["prompt_usage"].values())
        axs[0, 0].pie(prompt_values, labels=prompt_labels, autopct='%1.1f%%', startangle=90, textprops={'fontsize': 12})
        axs[0, 0].set_title('Prompt Type Distribution', fontsize=14, pad=10)

        # Quality Distribution
        quality_labels = list(data["quality_counts"].keys())
        quality_values = list(data["quality_counts"].values())
        axs[0, 1].bar(quality_labels, quality_values, color=['#2ecc71', '#e74c3c'])
        axs[0, 1].set_title('Conjecture Quality', fontsize=14, pad=10)
        axs[0, 1].tick_params(axis='y', labelsize=12)

        # Complexity Scores
        if data["complexity_scores"]:
            axs[1, 0].hist(data["complexity_scores"], bins=10, color='#3498db')
            axs[1, 0].set_title('Complexity Score Distribution', fontsize=14, pad=10)
            axs[1, 0].set_xlabel('Score', fontsize=12)
            axs[1, 0].set_ylabel('Frequency', fontsize=12)
        else:
            axs[1, 0].text(0.5, 0.5, 'No data', ha='center', va='center', fontsize=12)

        # Generation Time Trend
        if data["generation_times"]:
            indices = [item["index"] for item in data["generation_times"]]
            times = [item["time"] for item in data["generation_times"]]
            axs[1, 1].plot(indices, times, marker='o', linestyle='-', color='#9b59b6')
            axs[1, 1].set_title('Generation Time Trend', fontsize=14, pad=10)
            axs[1, 1].set_xlabel('Conjecture Index', fontsize=12)
            axs[1, 1].set_ylabel('Time (s)', fontsize=12)
        else:
            axs[1, 1].text(0.5, 0.5, 'No data', ha='center', va='center', fontsize=12)

        plt.tight_layout()
        plt.subplots_adjust(top=0.9)
        vis_path = self.output_dir / "visualizations.png"
        plt.savefig(vis_path)
        plt.close()
        return str(vis_path)

    def change_model(self, new_model_name):
        try:
            print(f"Loading new model: {new_model_name}")
            self.tokenizer = AutoTokenizer.from_pretrained(new_model_name)
            self.model = AutoModelForCausalLM.from_pretrained(new_model_name)
            self.generator = pipeline('text-generation', model=self.model, tokenizer=self.tokenizer)
            self.model_name = new_model_name
            return {"status": "success", "message": f"Model changed to {new_model_name}"}
        except Exception as e:
            return {"status": "error", "message": str(e)}

# Enhanced UI Components
def create_ui(generator):
    """Create a professional, mathematically themed Gradio UI"""

    def generate_conjecture_ui(prompt_type, custom_prompt, temperature, creativity):
        if prompt_type == "custom" and not custom_prompt:
            return "<div class='error'>Error: Please provide a custom prompt.</div>"
        result = generator.generate_conjecture(
            prompt_type=prompt_type if prompt_type != "custom" else None,
            custom_prompt=custom_prompt if prompt_type == "custom" else None,
            temperature=temperature,
            creativity=creativity
        )
        if "error" in result:
            return f"<div class='error'>Error: {result['error']}</div>"
        output = """
        <div class="conjecture-card">
            <h1>Generated Conjecture</h1>
        """
        if result.get("latex_formula"):
            output += f"""
            <div class="formula">
                {result['latex_formula']}
            </div>
            """
        output += f"""
            <div class="full-response">
                <h2>Details</h2>
                <p>{result['full_response']}</p>
            </div>
            <div class="quality-assessment">
                <h2>Assessment</h2>
                <ul>
                    <li>Quality: {result['validation']['overall_quality']}</li>
                    <li>Complexity: {result['validation']['complexity_score']}/100</li>
                    <li>Time: {result['generation_time']:.2f}s</li>
                </ul>
            </div>
        </div>
        """
        return output

    def batch_generate_ui(num_conjectures, selected_prompts, temperature, creativity):
        if not selected_prompts:
            return "<div class='error'>Error: Select at least one prompt type.</div>"
        results = generator.batch_generate(
            num_conjectures=int(num_conjectures),
            prompt_types=selected_prompts,
            temperature=temperature,
            creativity=creativity
        )
        if not results:
            return "<div class='error'>No conjectures generated.</div>"
        output = f"<h1>{len(results)} Conjectures</h1>"
        for i, result in enumerate(results, 1):
            output += f"""
            <div class="conjecture-card">
                <h2>Conjecture {i}</h2>
            """
            if result.get("latex_formula"):
                output += f"""
                <div class="formula">
                    {result['latex_formula']}
                </div>
                """
            output += f"""
                <p>{result['full_response']}</p>
                <p>Prompt: {result['prompt_type']}</p>
                <p>Quality: {result['validation']['overall_quality']}</p>
            </div>
            """
        return output

    def show_analytics():
        vis_path = generator.create_visualizations()
        analytics = generator.analyze_conjectures()
        if analytics["status"] != "success":
            return f"<div class='error'>{analytics['message']}</div>", None
        output = """
        <div class="analytics-dashboard">
            <div class="analytics-card">
                <h2>Summary</h2>
                <p>Total: {analytics['total_count']}</p>
                <p>Avg Time: {analytics['avg_generation_time']:.2f}s</p>
            </div>
            <div class="analytics-card">
                <h2>Quality</h2>
        """
        for q, c in analytics["quality_counts"].items():
            output += f"<p>{q}: {c}</p>"
        output += """
            </div>
            <div class="analytics-card">
                <h2>Prompts</h2>
        """
        for p, c in analytics["prompt_counts"].items():
            if c > 0:
                output += f"<p>{p}: {c}</p>"
        output += "</div></div>"
        return output, vis_path

    def show_learning_resource(concept):
        if concept in MATH_CONCEPTS:
            return f"""
            <div class="conjecture-card">
                <h1>{concept}</h1>
                <p>{MATH_CONCEPTS[concept]}</p>
            </div>
            """
        return "<div class='error'>Select a concept.</div>"

    def show_recent_conjectures():
        conjectures = generator.get_recent_conjectures()
        if not conjectures:
            return "<div class='error'>No recent conjectures.</div>"
        output = "<h1>Recent Conjectures</h1>"
        for i, c in enumerate(conjectures, 1):
            output += f"""
            <div class="conjecture-card">
                <h2>{i}. {c['date']}</h2>
            """
            if c.get("latex_formula"):
                output += f"""
                <div class="formula">
                    {c['latex_formula']}
                </div>
                """
            output += f"""
                <p>{c['full_response']}</p>
                <p>Prompt: {c['prompt_type']}</p>
                <p>Quality: {c['validation']['overall_quality']}</p>
            </div>
            """
        return output

    def change_model_ui(new_model):
        result = generator.change_model(new_model)
        return f"<div class='{'error' if result['status'] == 'error' else ''}'>{result['message']}</div>"

    with gr.Blocks(css=custom_css) as app:
        gr.HTML("""
        <div class="hero">
            <h1>Neural Conjecture Proposer</h1>
            <p>Where AI Meets Mathematical Innovation</p>
            <div class="math-decoration" style="top: 20px; left: 20px;">\\(\\sum_{n=1}^{\\infty} \\frac{1}{n^2} = \\frac{\\pi^2}{6}\\)</div>
            <div class="math-decoration" style="bottom: 20px; right: 20px;">\\(e^{i\\pi} + 1 = 0\\)</div>
            <div class="math-decoration" style="top: 50%; left: 50%; transform: translate(-50%, -50%);">\\(\\phi = \\frac{1 + \\sqrt{5}}{2}\\)</div>
        </div>
        """)

        with gr.Tabs():
            with gr.TabItem("Generate"):
                with gr.Group(elem_classes="tab-content"):
                    with gr.Row():
                        with gr.Column(scale=2):
                            with gr.Group(elem_classes="input-group"):
                                prompt_type = gr.Radio(
                                    label="Prompt Type",
                                    choices=["few-shot", "symbolic-pattern", "sequence", "reverse-engineering", "creative", "custom"],
                                    value="few-shot"
                                )
                                custom_prompt = gr.Textbox(label="Custom Prompt", lines=8, placeholder="Enter custom prompt here...")
                        with gr.Column(scale=1):
                            with gr.Group(elem_classes="input-group"):
                                temperature = gr.Slider(0.1, 1.5, 0.9, label="Temperature", step=0.1, elem_classes="custom-slider")
                                creativity = gr.Slider(0.1, 1.0, 0.8, label="Creativity", step=0.1, elem_classes="custom-slider")
                                generate_btn = gr.Button("Generate")
                    output_box = gr.Markdown()
                    generate_btn.click(generate_conjecture_ui, [prompt_type, custom_prompt, temperature, creativity], output_box)

            with gr.TabItem("Batch Generate"):
                with gr.Group(elem_classes="tab-content"):
                    with gr.Row():
                        with gr.Column(scale=2):
                            with gr.Group(elem_classes="input-group"):
                                num_conjectures = gr.Slider(1, 10, 3, label="Number of Conjectures", step=1, elem_classes="custom-slider")
                                selected_prompts = gr.CheckboxGroup(
                                    label="Prompt Types",
                                    choices=list(PROMPT_TEMPLATES.keys()),
                                    value=["few-shot", "symbolic-pattern"]
                                )
                        with gr.Column(scale=1):
                            with gr.Group(elem_classes="input-group"):
                                batch_temperature = gr.Slider(0.1, 1.5, 0.9, label="Temperature", step=0.1, elem_classes="custom-slider")
                                batch_creativity = gr.Slider(0.1, 1.0, 0.8, label="Creativity", step=0.1, elem_classes="custom-slider")
                                batch_btn = gr.Button("Generate Batch")
                    batch_output = gr.Markdown()
                    batch_btn.click(batch_generate_ui, [num_conjectures, selected_prompts, batch_temperature, batch_creativity], batch_output)

            with gr.TabItem("Analytics"):
                with gr.Group(elem_classes="tab-content"):
                    analytics_btn = gr.Button("Show Analytics")
                    with gr.Row():
                        analytics_output = gr.Markdown()
                        vis_img = gr.Image()
                    analytics_btn.click(show_analytics, [], [analytics_output, vis_img])

            with gr.TabItem("Recent"):
                with gr.Group(elem_classes="tab-content"):
                    recent_btn = gr.Button("Show Recent")
                    recent_output = gr.Markdown()
                    recent_btn.click(show_recent_conjectures, [], recent_output)

            with gr.TabItem("Learn"):
                with gr.Group(elem_classes="tab-content"):
                    with gr.Row():
                        with gr.Column(scale=1):
                            with gr.Group(elem_classes="input-group"):
                                concept_choice = gr.Radio(label="Concept", choices=list(MATH_CONCEPTS.keys()), value="Summation")
                                learn_btn = gr.Button("Show Details")
                        with gr.Column(scale=2):
                            concept_output = gr.Markdown()
                    learn_btn.click(show_learning_resource, concept_choice, concept_output)

            with gr.TabItem("Settings"):
                with gr.Group(elem_classes="tab-content"):
                    gr.HTML("<h2>Model Settings</h2>")
                    model_choice = gr.Dropdown(label="Model", choices=[DEFAULT_MODEL] + ALTERNATIVE_MODELS, value=generator.model_name)
                    change_btn = gr.Button("Change Model")
                    model_status = gr.Markdown()
                    change_btn.click(change_model_ui, model_choice, model_status)

        gr.HTML("""
        <div class="footer">
            <p>Neural Conjecture Proposer - Enhanced Edition</p>
            <p>Discovering Tomorrow's Theorems Today</p>
        </div>
        """)

    return app

def run_in_colab():
    print("Setting up in Google Colab...")
    try:
        import IPython
        IPython.get_ipython().system('pip install -q gradio matplotlib pandas tqdm transformers torch')
        generator = ConjectureGenerator()
        app = create_ui(generator)
        app.launch(share=True, height=1000)
    except Exception as e:
        print(f"Error: {e}")

def main():
    parser = argparse.ArgumentParser(description="Neural Conjecture Proposer")
    parser.add_argument("--model", type=str, default=DEFAULT_MODEL)
    parser.add_argument("--colab", action="store_true")
    args, _ = parser.parse_known_args()
    if args.colab:
        run_in_colab()
        return
    generator = ConjectureGenerator(model_name=args.model)
    app = create_ui(generator)
    app.launch()

if __name__ == "__main__":
    main()

Loading model: distilgpt2


Device set to use cuda:0


Model distilgpt2 loaded successfully!
It looks like you are running Gradio on a hosted a Jupyter notebook. For the Gradio app to work, sharing must be enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2d5bfda06139badbb2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
